# Lesson 3.3 — Named Templates & Helpers

**Phase 3: Advanced Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Define reusable named templates in `_helpers.tpl`
- Call named templates with `template` and `include`
- Understand the DRY principle applied to Helm templates
- Use `include` + `nindent` for safe YAML indentation

---

## Why Named Templates?

Without named templates you end up copy-pasting the same label block across every resource in your chart:

```yaml
# In deployment.yaml
labels:
  app: myapp
  release: my-release
  chart: myapp-0.1.0

# In service.yaml — identical!
labels:
  app: myapp
  release: my-release
  chart: myapp-0.1.0
```

Named templates (also called *partials* or *helpers*) solve this. You define once, call everywhere.

> **Reference:** [Named Templates](https://helm.sh/docs/chart_template_guide/named_templates/) — https://helm.sh/docs/chart_template_guide/named_templates/

---

## The `_helpers.tpl` Convention

Files that start with `_` in `templates/` are **not rendered** as K8s resources. They exist only to define named templates that other files can call.

By convention, all your helpers go in `_helpers.tpl`. Multiple `_` files are allowed.

---

## Setup

In [ ]:
rm -rf /tmp/helpersdemo && mkdir -p /tmp/helpersdemo/templates

cat > /tmp/helpersdemo/Chart.yaml << 'EOF'
apiVersion: v2
name: helpersdemo
version: 0.3.0
appVersion: "2.1.0"
EOF

cat > /tmp/helpersdemo/values.yaml << 'EOF'
replicaCount: 1
nameOverride: ""
fullnameOverride: ""
image:
  repository: nginx
  tag: "1.25"
service:
  port: 80
EOF

echo "Setup complete"

---

## Defining Named Templates

Use `{{- define "<chart-name>.<template-name>" -}}` ... `{{- end }}` to define a named template.

**Convention:** Always prefix with your chart name to avoid collisions with dependency charts.

In [ ]:
cat > /tmp/helpersdemo/templates/_helpers.tpl << 'EOF'
{{/*
  helpersdemo.fullname
  Generates a fully qualified app name.
  Truncated at 63 chars because Kubernetes DNS name limits.
*/}}
{{- define "helpersdemo.fullname" -}}
{{- if .Values.fullnameOverride }}
{{- .Values.fullnameOverride | trunc 63 | trimSuffix "-" }}
{{- else }}
{{- $name := default .Chart.Name .Values.nameOverride }}
{{- if contains $name .Release.Name }}
{{- .Release.Name | trunc 63 | trimSuffix "-" }}
{{- else }}
{{- printf "%s-%s" .Release.Name $name | trunc 63 | trimSuffix "-" }}
{{- end }}
{{- end }}
{{- end }}


{{/*
  helpersdemo.labels
  Common labels applied to all resources.
*/}}
{{- define "helpersdemo.labels" -}}
helm.sh/chart: {{ printf "%s-%s" .Chart.Name .Chart.Version | replace "+" "_" | trunc 63 | trimSuffix "-" }}
app.kubernetes.io/name: {{ .Chart.Name }}
app.kubernetes.io/instance: {{ .Release.Name }}
app.kubernetes.io/version: {{ .Chart.AppVersion | quote }}
app.kubernetes.io/managed-by: {{ .Release.Service }}
{{- end }}


{{/*
  helpersdemo.selectorLabels
  Selector labels — must be stable across upgrades (don't include chart version).
*/}}
{{- define "helpersdemo.selectorLabels" -}}
app.kubernetes.io/name: {{ .Chart.Name }}
app.kubernetes.io/instance: {{ .Release.Name }}
{{- end }}
EOF

echo "_helpers.tpl written"

---

## Calling Named Templates: `include` vs `template`

| Call | Returns | Can pipe to functions? |
|---|---|---|
| `{{ template "name" . }}` | string | No |
| `{{ include "name" . }}` | string | **Yes** |

**Always prefer `include`** — it lets you pipe the result through `nindent`, `indent`, `trim`, etc.

In [ ]:
cat > /tmp/helpersdemo/templates/deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  # include the fullname helper — pass . (current scope) as context
  name: {{ include "helpersdemo.fullname" . }}
  labels:
    # include labels and indent 4 spaces with nindent
    {{- include "helpersdemo.labels" . | nindent 4 }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      {{- include "helpersdemo.selectorLabels" . | nindent 6 }}
  template:
    metadata:
      labels:
        {{- include "helpersdemo.selectorLabels" . | nindent 8 }}
    spec:
      containers:
        - name: {{ .Chart.Name }}
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
EOF

cat > /tmp/helpersdemo/templates/service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: {{ include "helpersdemo.fullname" . }}
  labels:
    {{- include "helpersdemo.labels" . | nindent 4 }}
spec:
  selector:
    {{- include "helpersdemo.selectorLabels" . | nindent 4 }}
  ports:
    - port: {{ .Values.service.port }}
      protocol: TCP
EOF

helm template my-release /tmp/helpersdemo

---

## Passing Data to Named Templates

You can pass any object as context — not just `.`. A common pattern is passing a dictionary:

In [ ]:
cat >> /tmp/helpersdemo/templates/_helpers.tpl << 'EOF'

{{/*
  helpersdemo.containerImage
  Builds a full image string from a dict with 'repository' and 'tag'.
  Usage: include "helpersdemo.containerImage" .Values.image
*/}}
{{- define "helpersdemo.containerImage" -}}
{{- printf "%s:%s" .repository .tag -}}
{{- end }}
EOF

# Update the deployment to use the new helper
cat > /tmp/helpersdemo/templates/deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "helpersdemo.fullname" . }}
  labels:
    {{- include "helpersdemo.labels" . | nindent 4 }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      {{- include "helpersdemo.selectorLabels" . | nindent 6 }}
  template:
    metadata:
      labels:
        {{- include "helpersdemo.selectorLabels" . | nindent 8 }}
    spec:
      containers:
        - name: {{ .Chart.Name }}
          # Pass .Values.image as the context — the helper receives it as '.'
          image: {{ include "helpersdemo.containerImage" .Values.image }}
EOF

helm template my-release /tmp/helpersdemo --show-only templates/deployment.yaml

---

## The `fullname` Pattern Explained

The `fullname` helper generates a safe, unique resource name. Here's why:

- If `fullnameOverride` is set, use it directly (user wants full control)
- Otherwise combine `Release.Name` + chart name
- But if the release name already contains the chart name (e.g., release `myapp`, chart `myapp`), avoid duplication like `myapp-myapp`
- Truncate to 63 characters for Kubernetes DNS limits

This is the standard pattern from `helm create` — you will find it in almost every real-world chart.

---

## Exercise

In [ ]:
# TASK 1: Add a named template 'helpersdemo.serviceAccountName' that:
#   - Returns .Values.serviceAccount.name if set
#   - Otherwise returns the fullname
#   - Returns 'default' if serviceAccount.create is false
# TODO: add it to /tmp/helpersdemo/templates/_helpers.tpl


# TASK 2: Create a serviceaccount.yaml that:
#   - Only creates the resource if .Values.serviceAccount.create == true
#   - Uses the helpersdemo.fullname and helpersdemo.labels helpers
# TODO:


# TASK 3: Add a 'podAnnotations: {}' value and a named template
#         'helpersdemo.podAnnotations' that renders the annotations block
#         only if podAnnotations is non-empty.
#         Call it from the deployment template.
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Named templates live in `_helpers.tpl` (files starting with `_` are skipped by the renderer).
- Define with `{{- define "chart.name" -}}` ... `{{- end }}`.
- Always call with `include` (not `template`) so you can pipe to `nindent`.
- Name templates with a `chartname.` prefix to avoid collisions with sub-charts.
- The `fullname`, `labels`, and `selectorLabels` helpers from `helm create` are the standard starting point.

---

## Next Lesson

➡️ [Lesson 3.4 — Values Schema Validation](lesson-3.4-values-schema.ipynb)